# A - Architecture ablation (Cluster 3)

**Reviewer comments addressed**

* **R1.3 / R2.2 / R3.3** - the network (two hidden layers, uneven widths,
  tanh) is taken from prior work with no ablation; optimality is unproven.
* **R1.1 / R2.1 / R3.3 (novelty)** - the headline contribution is the
  *explicit-output* architecture, but it is never compared with the
  conventional `(u, v, phi)`-only PINN.

Two studies:

1. **Width / depth sweep** of the explicit model: `50-50`, `100-100`,
   `100-250` (the paper), `250-250`.
2. **Case A vs Case B** - the key one (the strategy says *"if you can only
   do one extra experiment, do this"*):
   * **Case A**: conventional PINN, outputs `(u, v, phi)`, equilibrium
     enforced with **second-order** derivatives.
   * **Case B**: this paper, outputs `(u, v, phi, sigma, D)`, **first-order**
     enforcement.

All variants are trained identically and scored against the FEM/your-FEM
reference for a fair comparison.

In [ ]:
# === Environment setup (robust: local / Colab native / VSCode-Colab) ===
# Run this cell FIRST. It is idempotent (safe to re-run) and does NOT touch
# Colab's preinstalled numpy/scipy/torch (avoids ABI breakage) - it just puts
# the repo's `src/` on the path and installs the one missing dep, scikit-fem.
import os, sys, subprocess, importlib
REPO_URL = 'https://github.com/Daniel14gonc/PINNs_piezoelectricity.git'

def _find_repo():
    """Return the repo root (folder with src/pinn_piezo) at/above cwd."""
    d = os.getcwd()
    for _ in range(10):
        if os.path.isdir(os.path.join(d, 'src', 'pinn_piezo')):
            return d
        nd = os.path.dirname(d)
        if nd == d:
            return None
        d = nd
    return None

repo = _find_repo()
if repo is None:
    # Fresh runtime: clone ONCE into a fixed absolute path (no nesting on re-run).
    base = '/content' if os.path.isdir('/content') else os.getcwd()
    repo = os.path.join(base, 'PINNs_piezoelectricity')
    if not os.path.isdir(os.path.join(repo, 'src', 'pinn_piezo')):
        subprocess.run(['git', 'clone', REPO_URL, repo], check=True)
    else:
        subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=False)

os.chdir(repo)
src = os.path.join(repo, 'src')
if src not in sys.path:
    sys.path.insert(0, src)   # import from source; NO pip install of the stack

# Only scikit-fem is missing on a stock Colab; install it (its deps already exist).
try:
    import skfem  # noqa: F401
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'scikit-fem'], check=True)

# Verify the revision modules made it into the checkout (i.e. you pushed them).
missing = [m for m in ('pinn_piezo', 'pinn_piezo.fem', 'pinn_piezo.metrics',
                       'pinn_piezo.indirect.standard')
           if importlib.util.find_spec(m) is None]
assert not missing, ('Missing modules: ' + ', '.join(missing) +
                     ' - push them to GitHub, then `git -C ' + repo +
                     ' pull` (or restart runtime) and re-run.')

import torch, pinn_piezo
print('pinn_piezo :', pinn_piezo.__file__)
print('repo       :', repo)
print('torch      :', torch.__version__, '| cuda:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Where this notebook writes its tables / figures.
from pathlib import Path
OUT = Path('outputs/revision'); OUT.mkdir(parents=True, exist_ok=True)
print('Artefacts ->', OUT.resolve())

In [ ]:
# The geometry/collocation datasets (data/*.npy) are gitignored, so a fresh
# clone does not have them. Regenerate the paper-size sets if missing
# (n_collocation=150 -> 150**2 = 22,500 collocation points).
import os
from pinn_piezo import geometry
for suffix in ('_m1', '_m1_d'):
    if not os.path.exists(f'data/xy_top_non_normalized{suffix}.npy'):
        geometry.generate_and_save(n_points=400, n_collocation=150,
                                   n_collocation_test=200, suffix=suffix,
                                   data_dir='data')
        print('generated data', suffix)
    else:
        print('data present', suffix)

In [ ]:
# Reference for scoring (your external FEM if available, else this repo's FEM).
import numpy as np, pandas as pd
from pinn_piezo import metrics, fem

# Reference fields from the validated scikit-fem solver (Notebook 00 shows it
# matches the analytical solution; poling_sign=-1 = this repo's indirect convention).
r = fem.solve_piezo('indirect', nx=300, ny=10, voltage=100.0, poling_sign=-1.0)
df = pd.DataFrame({'X_Coordinate': r.points[:,0], 'Y_Coordinate': r.points[:,1],
                   'X_Deflection': r.u, 'Y_Deflection': r.v, 'Potential': r.phi})
XY = df[['X_Coordinate','Y_Coordinate']].values
REF = {'u': df['X_Deflection'].values, 'v': df['Y_Deflection'].values, 'phi': df['Potential'].values}
print('reference points:', XY.shape[0])

In [ ]:
# Shared training data (indirect formulation, float64).
import torch
torch.set_default_dtype(torch.float64)
from pinn_piezo import geometry
from pinn_piezo.indirect import model as imodel, train as itrain
from pinn_piezo.indirect.train import tensorize

# Quick mode for a smoke test; set QUICK=False for paper-quality runs.
QUICK = True
import os
EP_ADAM  = int(os.environ.get('REV_ADAM',  200 if QUICK else 1000))
EP_LBFGS = int(os.environ.get('REV_LBFGS', 50  if QUICK else 200))

arrays = itrain.load_dataset('data', suffix='_m1', fraction=1.0)
tensors = itrain.to_device(arrays, DEVICE, dtype=torch.float64)
# Boundary coefficients (needed by the conventional baseline).
tensors['coeff_right'] = tensorize(geometry.build_coefficients(arrays['xy_right'][:, :2]), DEVICE)
tensors['coeff_left']  = tensorize(geometry.build_coefficients(arrays['xy_left'][:, :2]),  DEVICE)

def score(model, n_outputs):
    p = model(tensorize(XY, DEVICE)).detach().cpu().numpy()
    pred = {'u': p[:,0], 'v': p[:,1], 'phi': p[:,2]}
    return metrics.metrics_table(pred, REF)

## 1. Width / depth sweep (explicit model, Case B)

In [ ]:
ARCHS = [(50, 50), (100, 100), (100, 250), (250, 250)]
rows = []
for hs in ARCHS:
    print(f'\n=== training explicit model {hs} ===')
    torch.manual_seed(0); np.random.seed(0)
    model = imodel.build_default_model(device=DEVICE, model_type='pyramid', hidden_sizes=hs)
    res = itrain.train(model, tensors, epochs_adam=EP_ADAM, epochs_lbfgs=EP_LBFGS)
    mt = score(model, 8)
    rows.append({'arch': f'{hs[0]}-{hs[1]}',
                 'params': sum(p.numel() for p in model.parameters()),
                 'time_s': res['total_time'],
                 'L2_u': mt.loc['u','rel_L2'], 'L2_v': mt.loc['v','rel_L2'],
                 'L2_phi': mt.loc['phi','rel_L2']})
arch_table = pd.DataFrame(rows).set_index('arch')
arch_table.to_csv(OUT / 'ablation_width_depth.csv')
arch_table

*Read-out:* report this table and state that `100-250` is at least on par with
the alternatives (or, if another wins, adopt it). Either outcome answers
R1.3 / R2.2 / R3.3 - the choice is now *tested*, not asserted.

## 2. Case A (conventional) vs Case B (explicit) - the key experiment

In [ ]:
from pinn_piezo.indirect import standard

ARCH = (100, 250)
results = {}

# --- Case B: explicit outputs (this paper) ---
torch.manual_seed(0); np.random.seed(0)
mB = imodel.build_default_model(device=DEVICE, model_type='pyramid', hidden_sizes=ARCH)
resB = itrain.train(mB, tensors, epochs_adam=EP_ADAM, epochs_lbfgs=EP_LBFGS)
mtB = score(mB, 8)
results['Case B (explicit, 1st-order)'] = {'time_s': resB['total_time'],
    'final_loss': resB['loss_list'][-1], 'L2_u': mtB.loc['u','rel_L2'],
    'L2_v': mtB.loc['v','rel_L2'], 'L2_phi': mtB.loc['phi','rel_L2']}

# --- Case A: conventional (u,v,phi) with 2nd-order derivatives ---
torch.manual_seed(0); np.random.seed(0)
mA = standard.build_standard_model(device=DEVICE, hidden_sizes=ARCH)
resA = standard.train_standard(mA, tensors, epochs_adam=EP_ADAM, epochs_lbfgs=EP_LBFGS)
pA = mA(tensorize(XY, DEVICE)).detach().cpu().numpy()
mtA = metrics.metrics_table({'u':pA[:,0],'v':pA[:,1],'phi':pA[:,2]}, REF)
results['Case A (conventional, 2nd-order)'] = {'time_s': resA['total_time'],
    'final_loss': resA['loss_list'][-1], 'L2_u': mtA.loc['u','rel_L2'],
    'L2_v': mtA.loc['v','rel_L2'], 'L2_phi': mtA.loc['phi','rel_L2']}

ab = pd.DataFrame(results).T
ab.to_csv(OUT / 'ablation_caseA_vs_caseB.csv')
ab

This directly substantiates the central claim: explicit outputs let the PDEs
be enforced with first-order derivatives. Compare the two on **accuracy**
(relative L2), **training stability** (final loss / loss curve) and
**cost per step**. Whatever the numbers say, you now have evidence for the
architecture's value (or an honest, defensible statement of its trade-offs).

---
### Rebuttal snippet (Cluster 3)
> *We added an ablation over network width/depth (Table …) and, crucially, a
> direct comparison of our explicit-output formulation against the
> conventional `(u, v, phi)` PINN that enforces equilibrium with second-order
> derivatives (Table …). The explicit formulation [matches/improves] accuracy
> while [avoiding second derivatives / improving stability], substantiating
> the architectural contribution.*